# Distribución Poisson en Python

Este cuaderno está pensado para usarse en **Google Colab**. Aquí encontrarás:

- qué es la distribución Poisson,
- cómo calcular probabilidades como $P(X=k)$, $P(X\le k)$ y $P(X\ge k)$,
- cómo hallar la **media** y la **varianza**,
- cómo simular datos,
- cómo construir un **histograma**,
- y explicaciones paso a paso de lo que hace cada bloque de código.

---

## 1. Idea básica

La distribución Poisson se usa para modelar el número de veces que ocurre un evento en un intervalo fijo de tiempo, espacio, longitud, área, etc., cuando:

1. los eventos ocurren de manera aleatoria,
2. el promedio de ocurrencias es constante,
3. y las ocurrencias se consideran independientes.

Si $X \sim \text{Poisson}(\lambda)$, entonces:

$$
P(X=k)=\frac{e^{-\lambda}\lambda^k}{k!}, \qquad k=0,1,2,\dots
$$

donde $\lambda$ representa el número promedio de eventos en el intervalo.


In [ ]:
# Instalamos/importamos las librerías necesarias.
# scipy.stats nos ayuda con probabilidades teóricas.
# numpy se usa para cálculos numéricos y simulación.
# matplotlib permite hacer gráficas.

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson


## 2. Elegimos el parámetro $\lambda$

En este ejemplo tomaremos:

$$
\lambda = 4
$$

Esto significa que, en promedio, esperamos **4 eventos por intervalo**.

Por ejemplo, podría representar:

- 4 llamadas por minuto,
- 4 defectos por metro de material,
- 4 pacientes por hora,
- etc.


In [ ]:
# Parámetro de la distribución Poisson
lam = 4
print('Lambda =', lam)

## 3. Cálculo de probabilidades

Con `scipy.stats.poisson` podemos calcular distintas probabilidades.

### 3.1 Probabilidad puntual $P(X=k)$

La función `poisson.pmf(k, lam)` calcula la probabilidad de que ocurran exactamente `k` eventos.


In [ ]:
# Probabilidad de que ocurran exactamente k eventos
k = 3
p_igual = poisson.pmf(k, lam)

print(f'P(X = {k}) = {p_igual:.6f}')

### 3.2 Probabilidad acumulada $P(X\le k)$

La función `poisson.cdf(k, lam)` calcula la probabilidad acumulada:

$$
P(X \le k)
$$

Es decir, la probabilidad de observar **como máximo** `k` eventos.


In [ ]:
# Probabilidad de que ocurran como máximo k eventos
k = 3
p_menor_igual = poisson.cdf(k, lam)

print(f'P(X <= {k}) = {p_menor_igual:.6f}')

### 3.3 Probabilidad $P(X\ge k)$

Para calcular probabilidades del tipo

$$
P(X \ge k)
$$

podemos usar el complemento:

$$
P(X\ge k)=1-P(X\le k-1)
$$

También existe la función de supervivencia `poisson.sf(k-1, lam)`, que suele ser más estable numéricamente.


In [ ]:
# Probabilidad de que ocurran al menos k eventos
k = 5
p_mayor_igual = poisson.sf(k - 1, lam)

print(f'P(X >= {k}) = {p_mayor_igual:.6f}')

## 4. Tabla de varias probabilidades

A veces conviene ver muchas probabilidades a la vez. En el siguiente bloque calculamos $P(X=k)$ para varios valores de `k`.


In [ ]:
# Calculamos varias probabilidades puntuales
valores_k = np.arange(0, 11)
probabilidades = poisson.pmf(valores_k, lam)

print(' k   P(X=k)')
print('-' * 18)
for k, p in zip(valores_k, probabilidades):
    print(f'{k:2d}   {p:.6f}')

## 5. Media y varianza

Una propiedad muy importante de la distribución Poisson es que:

$$
\mathbb{E}[X]=\lambda, \qquad \mathrm{Var}(X)=\lambda
$$

Es decir, en Poisson la **media y la varianza son iguales**.

Con `scipy` podemos obtener estos valores directamente.


In [ ]:
# Media y varianza teóricas
media = poisson.mean(lam)
varianza = poisson.var(lam)

print(f'Media teórica     = {media:.4f}')
print(f'Varianza teórica  = {varianza:.4f}')

## 6. Simulación de datos

Ahora vamos a generar una muestra aleatoria con distribución Poisson usando `numpy.random.poisson`.

Esto no produce las probabilidades exactas teóricas, sino una **muestra simulada** que debería parecerse al comportamiento de la distribución cuando el tamaño de muestra es suficientemente grande.


In [ ]:
# Fijamos semilla para que el resultado sea reproducible
np.random.seed(42)

# Generamos 1000 observaciones de una Poisson(lam)
muestra = np.random.poisson(lam=lam, size=1000)

print('Primeros 20 datos simulados:')
print(muestra[:20])

## 7. Media y varianza de la muestra simulada

Aquí comparamos la teoría con los datos simulados. Como la muestra es aleatoria, los valores no coincidirán exactamente con $\lambda$, pero deberían estar cerca.


In [ ]:
# Media y varianza muestral
media_muestral = np.mean(muestra)
varianza_muestral = np.var(muestra, ddof=0)

print(f'Media muestral     = {media_muestral:.4f}')
print(f'Varianza muestral  = {varianza_muestral:.4f}')

## 8. Histograma de la muestra

El histograma muestra con qué frecuencia aparece cada número de eventos en la simulación.

Además, superponemos las probabilidades teóricas de Poisson para comparar:

- las barras representan la frecuencia observada en la simulación,
- los puntos muestran la probabilidad teórica.


In [ ]:
# Histograma + probabilidades teóricas
valores = np.arange(0, max(muestra) + 1)
pmf_teorica = poisson.pmf(valores, lam)

plt.figure(figsize=(10, 6))
plt.hist(muestra, bins=np.arange(-0.5, max(muestra)+1.5, 1), density=True, alpha=0.7, edgecolor='black')
plt.plot(valores, pmf_teorica, 'o-', linewidth=2)
plt.xlabel('Número de eventos')
plt.ylabel('Frecuencia relativa / Probabilidad')
plt.title(f'Histograma de datos simulados y distribución Poisson teórica (lambda={lam})')
plt.grid(alpha=0.3)
plt.show()

### Interpretación del histograma

Si el modelo Poisson es adecuado, el histograma de la muestra simulada debe parecerse a la forma de la distribución teórica.

En este caso, como $\lambda=4$:

- los valores cercanos a 4 suelen ser los más frecuentes,
- valores muy pequeños o muy grandes aparecen menos veces,
- la distribución es discreta, porque solo toma valores enteros: 0, 1, 2, 3, ...


## 9. Gráfica de la función de probabilidad

Además del histograma, también es útil graficar directamente la función de probabilidad teórica.


In [ ]:
# Gráfica de la función de probabilidad teórica
k_vals = np.arange(0, 15)
pmf_vals = poisson.pmf(k_vals, lam)

plt.figure(figsize=(10, 6))
plt.stem(k_vals, pmf_vals, basefmt=' ')
plt.xlabel('k')
plt.ylabel('P(X=k)')
plt.title(f'Función de probabilidad de una Poisson({lam})')
plt.grid(alpha=0.3)
plt.show()

## 10. Función reutilizable

En el siguiente bloque definimos una función que resume los cálculos principales para cualquier valor de $\lambda$ y cualquier valor de `k`.


In [ ]:
def resumen_poisson(lam, k):
    """
    Muestra un resumen básico para una distribución Poisson.
    lam: parámetro lambda
    k: valor entero para calcular probabilidades
    """
    print(f'Distribución: Poisson({lam})')
    print(f'P(X = {k})   = {poisson.pmf(k, lam):.6f}')
    print(f'P(X <= {k})  = {poisson.cdf(k, lam):.6f}')
    print(f'P(X >= {k})  = {poisson.sf(k-1, lam):.6f}')
    print(f'Media        = {poisson.mean(lam):.4f}')
    print(f'Varianza     = {poisson.var(lam):.4f}')

# Ejemplo de uso
resumen_poisson(lam=6, k=4)

## 11. Ejercicio propuesto

Prueba cambiar el valor de `lam` en los ejemplos anteriores y responde:

1. ¿Qué ocurre con la forma de la distribución cuando $\lambda$ aumenta?
2. ¿La media y la varianza siguen siendo iguales?
3. ¿Dónde parece concentrarse más la probabilidad?

Por ejemplo, compara:

- $\lambda=2$
- $\lambda=5$
- $\lambda=10$


## 12. Conclusión

En este cuaderno aprendiste a:

- calcular probabilidades con la distribución Poisson,
- obtener media y varianza,
- simular datos,
- construir un histograma,
- e interpretar la relación entre la muestra simulada y el modelo teórico.

Este material es útil para cursos introductorios de probabilidad y estadística, y también como base para aplicaciones reales en colas, llamadas, defectos, incidentes, pacientes, llegadas y conteos de eventos.
